In [ ]:
import pandas as pd

expr = pd.read_csv(
    "data/OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv",
    index_col=0
)

models = pd.read_csv("data/Model.csv")

prism = pd.read_csv(
    "data/secondary-screen-dose-response-curve-parameters.csv"
)

treatments = pd.read_csv(
    "data/secondary-screen-replicate-collapsed-treatment-info.csv"
)

cell_info = pd.read_csv(
    "data/secondary-screen-cell-line-info.csv"
)

print("Expression:")
print(expr.shape)
print(expr.index[:5])
print(expr.columns[:10])

print("\nModels:")
print(models.columns.tolist())

print("\nPRISM:")
print(prism.shape)
print(prism.columns.tolist())

print("\nTreatment:")
print(treatments.columns.tolist())

print("\nPRISM cell info:")
print(cell_info.columns.tolist())

In [ ]:
expr = expr.set_index("ModelID")

In [ ]:
genes = ["CDK4", "CDK6", "WEE1", "CHEK1", "SLC7A11"]


def find_gene_column(df, gene):
    matches = [
        col for col in df.columns
        if col == gene or col.startswith(gene + " ")
    ]

    if len(matches) == 0:
        print(f"WARNING: {gene} not found")
        return None

    if len(matches) > 1:
        print(f"WARNING: multiple matches for {gene}: {matches}")

    return matches[0]


gene_columns = {
    gene: find_gene_column(expr, gene)
    for gene in genes
}

print(gene_columns)

In [ ]:
gene_columns = {
    gene: col
    for gene, col in gene_columns.items()
    if col is not None
}

expr_targets = expr[list(gene_columns.values())].copy()

expr_targets.columns = [
    next(g for g, c in gene_columns.items() if c == col)
    for col in expr_targets.columns
]

expr_targets.index.name = "DepMap_ID"
expr_targets = expr_targets.reset_index()

print(expr_targets.head())

In [ ]:
print(
    models[
        [ "OncotreeLineage"]
    ].drop_duplicates().to_string()
)

In [ ]:
models_sub = models[
    models["OncotreeLineage"].isin([
        "Bowel",
        "Uterus"
    ])
].copy()
models_sub

In [ ]:
crc = models[
    models["OncotreePrimaryDisease"]
    .str.contains("Colorectal", case=False, na=False)
]

ucec = models[
    models["OncotreePrimaryDisease"]
    .str.contains(
        "Endometrial|Uterine",
        case=False,
        na=False
    )
]

models_selected = pd.concat([crc, ucec]).drop_duplicates("ModelID")
models_selected

In [ ]:
expr_targets

In [ ]:
expr_selected = expr_targets.merge(
    models_selected,
    left_on="DepMap_ID",
    right_on="ModelID",
    how="inner"
)

print(expr_selected.shape)

In [ ]:
print(treatments.columns.tolist())

In [ ]:
search_terms = [
    "palbociclib",
    "ribociclib",
    "abemaciclib",
    "adavosertib",
    "AZD1775",
    "MK-1775",
    "prexasertib",
    "MK-8776",
    "erastin",
    "sulfasalazine",
    "ferropt"
]

for term in search_terms:
    mask = treatments.astype(str).apply(
        lambda x: x.str.contains(term, case=False, na=False)
    ).any(axis=1)

    hits = treatments.loc[mask]

    if len(hits):
        print("\n###", term)
        print(hits.to_string(index=False))

In [ ]:
for term in ["CDK4", "CDK6", "WEE1", "CHEK1", "SLC7A11"]:
    mask = treatments.astype(str).apply(
        lambda x: x.str.contains(term, case=False, na=False)
    ).any(axis=1)

    hits = treatments.loc[mask]

    if len(hits):
        print("\n### TARGET:", term)
        print(hits.to_string(index=False))

In [ ]:
print(prism.columns.tolist())
print(prism.head().T)

In [ ]:
prism = prism.merge(
    cell_info[["ccle_name", "depmap_id"]],
    on="ccle_name",
    how="left"
)
prism

In [ ]:
data = prism.merge(
    expr_selected,
    left_on="depmap_id_x",
    right_on="DepMap_ID",
    how="inner"
)
data

In [ ]:
drug_tests = {
    "CDK4": [
        "palbociclib",
        "ribociclib",
        "abemaciclib"
    ],
    "CDK6": [
        "palbociclib",
        "ribociclib",
        "abemaciclib"
    ],
    "WEE1": [
        
        
        "MK-1775"
    ],
    "CHEK1": [
        
        
    ],
    "SLC7A11": [
        "erastin",
        "sulfasalazine"
    ]
}

In [ ]:
from scipy.stats import spearmanr
import numpy as np
from statsmodels.stats.multitest import multipletests


response_col = "auc"

results = []

for target, drugs in drug_tests.items():

    for drug in drugs:

        drug_df = prism[
            prism["name"].str.lower() == drug.lower()
        ].copy()

        if drug_df.empty:
            print(f"Drug not found: {drug}")
            continue

        df = expr_selected[
            ["DepMap_ID", target]
        ].merge(
            drug_df[
                ["depmap_id_x", "name", response_col]
            ],
            left_on="DepMap_ID",
            right_on="depmap_id_x",
            how="inner"
        )

        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna(subset=[target, response_col])

        if len(df) < 3:
            print(f"Skipping {target} - {drug}: n={len(df)}")
            continue

        rho, p = spearmanr(
            df[target],
            df[response_col]
        )

        results.append({
            "target": target,
            "drug": drug,
            "n": len(df),
            "rho": rho,
            "pvalue": p
        })


results = pd.DataFrame(results)

if not results.empty:
    results["FDR"] = multipletests(
        results["pvalue"],
        method="fdr_bh"
    )[1]

    results = results.sort_values("pvalue")

results